# Setup — Schemas, Volume + Initial Staging Load

One-time (idempotent) provisioning of the UC schemas + volume (single catalog) and a full
initial staging load of all 12 entities from `samples.tpch` (`batch_id = 1`). Safe to re-run.
In demos this can be run once up front, then skipped. The target catalog is assumed to exist.

In [ ]:
%run ./initialize

In [ ]:
# Single catalog (assumed to exist) — create one schema per layer / bronze source.
# Staging schema + volume
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{staging_schema}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{staging_schema}`.`{staging_volume}`")

# Bronze: one schema per source system
for system in bronze_source_systems:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{bronze_schema(system)}`")

# Silver + gold conformed schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{silver_schema}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{gold_schema}`")

print("Schemas and volume provisioned.")

In [ ]:
# Reset the staging source tree so the initial load is clean / idempotent
dbutils.fs.rm(sources_root, True)
print(f"Cleared {sources_root}")

In [ ]:
# Full initial staging load (batch_id = 1) — all entities shredded into source-system folders
batch_id = 1
load_ts = datetime.now()

# Business effective date for the SCD2 customer/supplier dimensions. The baseline version is
# effective from the dataset floor (1 Jan 1992) so it covers every historical order; later runs
# introduce versions with newer (or backdated) effective dates. Facts resolve the version
# effective as of the order_date (point-in-time). See run_2 / run_3 for the change dates.
BASELINE_EFFECTIVE = "1992-01-01 00:00:00"


def load(sql, system, entity, effective_date=None, scd2=False):
    df = with_metadata(spark.sql(sql), system, batch_id, load_ts)
    if effective_date is not None:
        df = with_effective(df, effective_date)
    # SCD2 dimension feeds carry a CDC cdc_operation flag (all 'U' on the baseline) so later runs
    # can issue deletes via cdcSettings.apply_as_deletes. Facts / SCD1 reference data do not.
    if scd2:
        df = with_op(df, "U")
    write_landing(df, system, entity, INITIAL_ZONE, load_ts)


# Reference data (SCD1 — static lookups, no cdc_operation flag)
load(f"SELECT r_regionkey, r_name, r_comment FROM {sample_source_schema}.region", "reference_data", "region")
load(f"SELECT n_nationkey, n_name, n_regionkey, n_comment FROM {sample_source_schema}.nation", "reference_data", "nation")

# CRM (customer master shredded across 3 entities) — SCD2 dims carry a business effective_date
load(f"SELECT c_custkey AS customer_id, c_name AS name, c_acctbal AS acctbal, c_mktsegment AS mktseg FROM {sample_source_schema}.customer", "crm", "customer", BASELINE_EFFECTIVE, scd2=True)
load(f"SELECT c_custkey AS customer_id, c_address AS address, c_nationkey AS nat_id FROM {sample_source_schema}.customer", "crm", "customer_address", BASELINE_EFFECTIVE, scd2=True)
load(f"SELECT c_custkey AS customer_id, 'M' AS ptype, c_phone AS phone FROM {sample_source_schema}.customer", "crm", "customer_phone", BASELINE_EFFECTIVE, scd2=True)

# Procurement + vendor management (supplier master shredded) — supplier is SCD2 with effective_date
load(f"SELECT s_suppkey AS supplier_id, s_name AS name, s_acctbal AS acctbal, s_comment AS comment FROM {sample_source_schema}.supplier", "procurement", "supplier", BASELINE_EFFECTIVE, scd2=True)
load(f"SELECT s_suppkey AS supplier_id, s_address AS address, s_nationkey AS nat_id FROM {sample_source_schema}.supplier", "vendor_mgmt", "supplier_address", scd2=True)
load(f"SELECT s_suppkey AS supplier_id, s_phone AS phone FROM {sample_source_schema}.supplier", "vendor_mgmt", "supplier_phone", scd2=True)

# Order management + fulfillment (facts)
load(f"SELECT o_orderkey, o_custkey, o_orderstatus, o_totalprice, o_orderdate, o_orderpriority, o_clerk, o_shippriority, o_comment FROM {sample_source_schema}.orders", "order_mgmt", "orders")
load(f"SELECT l_orderkey, l_partkey, l_suppkey, l_linenumber, l_quantity, l_extendedprice, l_discount, l_tax, l_returnflag, l_linestatus, l_shipdate, l_commitdate, l_receiptdate, l_shipinstruct, l_shipmode, l_comment FROM {sample_source_schema}.lineitem", "order_fulfillment", "lineitem")

# Product catalog + inventory — part is SCD2 (carries cdc_operation); partsupp is an append fact
load(f"SELECT p_partkey, p_name, p_mfgr, p_brand, p_type, p_size, p_container, p_retailprice, p_comment FROM {sample_source_schema}.part", "product_catalog", "part", scd2=True)
load(f"SELECT ps_partkey, ps_suppkey, ps_availqty, ps_supplycost, ps_comment FROM {sample_source_schema}.partsupp", "inventory", "partsupp")

print("Initial staging load complete (batch_id = 1)")